In [1]:

import os
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, models

# Corpus lives in data/ alongside the generated indexes
CORPUS_PATH = "data/bible_corpus.csv"

# ── Model definitions ──────────────────────────────────────────────────────────
# Each entry: (output_dir, callable_that_returns_a_SentenceTransformer)

def build_ancient_greek_biblical_sbert():
    """ancient-greek-biblical-sbert: Transformer + mean pooling (no internal normalize)."""
    word_embedding_model = models.Transformer("models/ancient-greek-biblical-sbert")
    pooling_model = models.Pooling(
        word_embedding_model.get_word_embedding_dimension(),
        pooling_mode="mean",
    )
    return SentenceTransformer(modules=[word_embedding_model, pooling_model])

def build_greek_nt_sbert_v2():
    """greek-nt-sbert_v2: Transformer + CLS pooling + Dense + Normalize (full ST model)."""
    return SentenceTransformer("models/greek-nt-sbert_v2")

MODEL_CONFIGS = [
    ("ancient-greek-biblical-sbert", build_ancient_greek_biblical_sbert),
    ("greek-nt-sbert_v2",            build_greek_nt_sbert_v2),
]

# ── Load corpus ────────────────────────────────────────────────────────────────
print("Loading corpus...")
corpus_df = pd.read_csv(CORPUS_PATH)
sentences = corpus_df["text"].tolist()
print(f"Corpus size: {len(sentences)} verses")

# ── Build one index per model ──────────────────────────────────────────────────
for model_name, model_factory in MODEL_CONFIGS:
    out_dir = os.path.join("data", model_name)
    os.makedirs(out_dir, exist_ok=True)

    index_path    = os.path.join(out_dir, "bible_greek.index")
    metadata_path = os.path.join(out_dir, "bible_metadata.pkl")

    print(f"\n[{model_name}] Building FAISS index → {index_path}")

    model = model_factory()

    print(f"[{model_name}] Encoding {len(sentences)} verses...")
    embeddings = model.encode(
        sentences,
        show_progress_bar=True,
        convert_to_numpy=True,
        batch_size=64,
    )

    # Normalize to unit length so IndexFlatIP == cosine similarity.
    # (greek-nt-sbert_v2 already normalises internally; this is a no-op for it.)
    faiss.normalize_L2(embeddings)

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings)

    faiss.write_index(index, index_path)
    corpus_df.to_pickle(metadata_path)

    print(f"[{model_name}] Done — {index.ntotal} vectors, dim={dimension}")

    # Free GPU/CPU memory before next model
    del model, embeddings, index

print("\nAll indexes built successfully.")


/home/jon/Documents/codespace/projects/greek-semantic-ui/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading corpus...
Corpus size: 38526 verses

[ancient-greek-biblical-sbert] Building FAISS index → data/ancient-greek-biblical-sbert/bible_greek.index


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2427.03it/s, Materializing param=pooler.dense.weight]                               


[ancient-greek-biblical-sbert] Encoding 38526 verses...


Batches: 100%|██████████| 602/602 [00:50<00:00, 11.94it/s]


[ancient-greek-biblical-sbert] Done — 38526 vectors, dim=768

[greek-nt-sbert_v2] Building FAISS index → data/greek-nt-sbert_v2/bible_greek.index


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2138.57it/s, Materializing param=pooler.dense.weight]                               


[greek-nt-sbert_v2] Encoding 38526 verses...


Batches: 100%|██████████| 602/602 [01:20<00:00,  7.52it/s]


[greek-nt-sbert_v2] Done — 38526 vectors, dim=768

All indexes built successfully.


In [2]:

# ── Quick smoke-test ───────────────────────────────────────────────────────────
import faiss, pandas as pd
from sentence_transformers import SentenceTransformer

def smoke_test(model_name: str, query: str = "ἀγάπη", top_k: int = 5):
    index    = faiss.read_index(f"data/{model_name}/bible_greek.index")
    metadata = pd.read_pickle(f"data/{model_name}/bible_metadata.pkl")
    model    = SentenceTransformer(f"models/{model_name}") \
               if model_name != "ancient-greek-biblical-sbert" \
               else _load_agbs()

    vec = model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(vec)
    distances, indices = index.search(vec, top_k)
    results = metadata.iloc[indices[0]].copy()
    results["similarity"] = distances[0]
    print(f"\n── {model_name} | query: '{query}' ──")
    print(results[["verse_ref", "text", "similarity"]].to_string(index=False))

def _load_agbs():
    from sentence_transformers import models
    wem = models.Transformer("models/ancient-greek-biblical-sbert")
    pm  = models.Pooling(wem.get_word_embedding_dimension(), pooling_mode="mean")
    from sentence_transformers import SentenceTransformer as ST
    return ST(modules=[wem, pm])

smoke_test("ancient-greek-biblical-sbert")
smoke_test("greek-nt-sbert_v2")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2359.95it/s, Materializing param=pooler.dense.weight]                               



── ancient-greek-biblical-sbert | query: 'ἀγάπη' ──
          verse_ref                                                                                                                                           text  similarity
         1 john 3:1         ἴδετε ποταπὴν ἀγάπην δέδωκεν ἡμῖν ὁ πατήρ, ἵνα τέκνα θεοῦ κληθῶμεν καὶ ἐσμέν. διὰ τοῦτο ὁ κόσμος οὐ γινώσκει ἡμᾶς, ὅτι οὐκ ἔγνω αὐτόν.    0.930523
1 corinthians 16:14                                                                                                                   πάντα ὑμῶν ἐν ἀγάπῃ γινέσθω.    0.929476
         john 15:13                                                            μείζονα ταύτης ἀγάπην οὐδεὶς ἔχει ἵνα τις τὴν ψυχὴν αὐτοῦ θῇ ὑπὲρ τῶν φίλων αὐτοῦ.    0.926110
        1 peter 4:8                                                       πρὸ πάντων δὲ τὴν εἰς ἑαυτοὺς ἀγάπην ἐκτενῆ ἔχοντες, ὅτι ἀγάπη καλύπτει πλῆθος ἁμαρτιῶν.    0.925158
        1 john 4:18 φόβος οὐκ ἔστιν ἐν τῇ ἀγάπῃ, ἀλλ᾽ ἡ τελεία ἀγάπη ἔξ

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2335.95it/s, Materializing param=pooler.dense.weight]                               



── greek-nt-sbert_v2 | query: 'ἀγάπη' ──
        verse_ref                                                  text  similarity
       Odes 14:17                                     ὁ ἀμνὸς τοῦ θεοῦ,    0.688125
       Odes 14:11                                            ἐπουράνιε,    0.675933
        Odes 14:1                                  Δόξα ἐν ὑψίστοις θεῷ    0.673925
    Proverbs 27:5 κρείσσους ἔλεγχοι ἀποκεκαλυμμένοι κρυπτομένης φιλίας.    0.659751
1 Chronicles 1:24                                                 Σαλα,    0.657270


In [3]:

# ── Concept Bank ──────────────────────────────────────────────────────────────
# Build a bank of English theological/biblical concepts encoded with
# greek-nt-sbert_v2 so we can do cross-lingual concept retrieval against
# any Greek verse.

import os
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

CONCEPTS = {
    "Core Theology": [
        "faith", "grace", "salvation", "redemption", "atonement",
        "justification by faith", "sanctification", "glorification",
        "predestination", "election", "repentance", "conversion",
        "new birth", "regeneration", "adoption as children of God",
    ],
    "God's Character": [
        "divine love", "holiness of God", "righteousness of God", "mercy",
        "compassion", "wrath of God", "divine justice", "sovereignty of God",
        "faithfulness of God", "goodness of God", "truth", "omnipotence",
        "eternal nature of God", "glory of God",
    ],
    "Christology": [
        "incarnation of Christ", "crucifixion", "resurrection of Christ",
        "ascension of Christ", "second coming of Christ", "Son of God",
        "Lord and Savior", "High Priest", "mediator between God and man",
        "Lamb of God", "Messiah", "King of kings", "suffering servant",
        "bread of life", "light of the world", "Good Shepherd",
        "Jesus as teacher", "authority of Jesus", "I AM sayings",
    ],
    "Holy Spirit": [
        "Holy Spirit", "spiritual gifts", "fruit of the Spirit",
        "baptism of the Spirit", "divine intercession", "conviction of sin",
        "spiritual power", "inspired by God", "spiritual renewal",
    ],
    "Eschatology": [
        "final judgment", "eternal life", "heaven", "hell",
        "kingdom of God", "resurrection of the dead", "new creation",
        "end times", "eternal punishment", "Day of the Lord",
        "eternal reward", "cosmic restoration", "judgment of nations",
        "signs of the end", "woes and tribulation",
    ],
    "Sin and Corruption": [
        "sin", "pride", "greed", "envy", "lust", "anger", "idolatry",
        "disobedience", "temptation", "corruption", "wickedness",
        "spiritual blindness", "unbelief", "hypocrisy",
        "hardness of heart", "spiritual death", "shame and disgrace",
    ],
    "Virtues and Ethics": [
        "love of neighbor", "humility", "generosity", "honesty", "courage",
        "patience", "self-control", "integrity", "wisdom", "discernment",
        "obedience to God", "compassion for the poor", "peacemaking",
        "servanthood", "contentment", "perseverance",
    ],
    "Covenant and Law": [
        "covenant with God", "the Law of Moses", "the commandments",
        "divine promise", "fulfillment of prophecy", "sacrifice and offering",
        "priesthood", "Passover", "circumcision", "the Sabbath",
        "new covenant", "old covenant", "Torah",
        "Pharisees and religious leaders", "temple worship",
    ],
    "Spiritual Practices": [
        "prayer", "worship", "fasting", "thanksgiving", "praise to God",
        "confession of sin", "baptism", "the Lord's Supper", "Scripture",
        "meditation on God's word", "devotion to God", "spiritual discipline",
        "witnessing and testimony",
    ],
    "Community and Mission": [
        "the Church", "fellowship of believers", "discipleship", "evangelism",
        "unity in Christ", "reconciliation", "hospitality",
        "the Great Commission", "spiritual leadership",
        "bearing one another's burdens", "love for the church",
        "apostolic mission", "sending and being sent",
    ],
    "Biblical History": [
        "creation of the world", "the fall of humanity", "the Exodus",
        "wilderness wandering", "the promised land", "the Davidic kingdom",
        "prophetic vision", "exile and restoration",
        "fulfillment of the law", "history of Israel",
        "the patriarchs", "Abraham and the covenant",
    ],
    "Spiritual Conflict": [
        "spiritual warfare", "the devil", "demonic possession",
        "overcoming evil", "armor of God", "spiritual deception",
        "the world and its temptations", "testing and trials",
        "exorcism", "binding the strong man",
    ],
    "Miracles and Signs": [
        "healing the sick", "casting out demons", "raising the dead",
        "feeding the multitude", "walking on water", "turning water to wine",
        "signs and wonders", "miraculous catch of fish",
        "cleansing of leprosy", "restoring sight to the blind",
        "power over nature", "divine provision",
    ],
    "Angels and Messengers": [
        "angels", "angel of the Lord", "divine messenger",
        "heavenly host", "angelic announcement", "vision from God",
        "voice from heaven", "the Holy Spirit descending",
    ],
    "Marriage and Family": [
        "marriage", "husband and wife", "divorce", "adultery",
        "children and parents", "honoring father and mother",
        "the family of God", "widows and orphans",
        "submission in the household", "love in marriage",
    ],
    "Passion and Crucifixion": [
        "betrayal of Jesus", "trial before Pilate", "carrying the cross",
        "crucifixion of Jesus", "the thieves on the cross",
        "death of Jesus", "burial of Jesus",
        "mocking of Jesus", "crown of thorns",
    ],
    "Women in the Narrative": [
        "Mary Magdalene", "women at the tomb", "the Samaritan woman",
        "Mary and Martha", "the persistent widow", "a woman's faith",
        "anointing of Jesus by a woman", "women disciples of Jesus",
    ],
}

# Flatten into a DataFrame
rows = [
    {"concept": concept, "category": category}
    for category, concepts in CONCEPTS.items()
    for concept in concepts
]
concepts_df = pd.DataFrame(rows)
print(f"Concept bank: {len(concepts_df)} entries across {len(CONCEPTS)} categories\n")
print(concepts_df["category"].value_counts().to_string())

# Encode with greek-nt-sbert_v2 (cross-lingual: English concepts map into
# the same vector space as Ancient Greek text)
model = SentenceTransformer("models/greek-nt-sbert_v2")

print("\nEncoding concepts...")
embeddings = model.encode(
    concepts_df["concept"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64,
)

# The model's Normalize layer outputs unit vectors; call again as a safety measure
faiss.normalize_L2(embeddings)

out_dir = "data/greek-nt-sbert_v2"
os.makedirs(out_dir, exist_ok=True)
concepts_df.to_pickle(os.path.join(out_dir, "concept_bank.pkl"))
np.save(os.path.join(out_dir, "concept_embeddings.npy"), embeddings)

print(f"\nSaved -> {out_dir}/concept_bank.pkl  ({len(concepts_df)} concepts)")
print(f"Saved -> {out_dir}/concept_embeddings.npy  shape={embeddings.shape}")


Concept bank: 215 entries across 17 categories

category
Christology                19
Sin and Corruption         17
Virtues and Ethics         16
Core Theology              15
Eschatology                15
Covenant and Law           15
God's Character            14
Spiritual Practices        13
Community and Mission      13
Miracles and Signs         12
Biblical History           12
Marriage and Family        10
Spiritual Conflict         10
Holy Spirit                 9
Passion and Crucifixion     9
Angels and Messengers       8
Women in the Narrative      8


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2254.33it/s, Materializing param=pooler.dense.weight]                               



Encoding concepts...


Batches: 100%|██████████| 4/4 [00:00<00:00, 54.67it/s]


Saved -> data/greek-nt-sbert_v2/concept_bank.pkl  (215 concepts)
Saved -> data/greek-nt-sbert_v2/concept_embeddings.npy  shape=(215, 768)


In [4]:

# ── Organic Vocabulary (ESV TF-IDF) ──────────────────────────────────────────
# Run cell 5 (ESV parser) first to generate data/esv.csv, then run this cell.
# TF-IDF on the ESV corpus extracts statistically discriminative modern English
# terms without any hand-curation.  The cross-lingual model determines
# relevance purely through embedding geometry.

import os
import numpy as np
import pandas as pd
import faiss
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sentence_transformers import SentenceTransformer

ESV_CSV = "data/esv.csv"
if not os.path.exists(ESV_CSV):
    raise FileNotFoundError(
        f"{ESV_CSV} not found — run cell 5 (ESV parser) first."
    )

esv_df = pd.read_csv(ESV_CSV)
verses = esv_df["verse_text"].dropna().tolist()
print(f"ESV corpus: {len(verses)} verses")

# ── Additional domain stopwords ────────────────────────────────────────────────
# Pronouns, biblical conjunctions, and high-frequency grammatical words that
# survive the max_df filter but carry no semantic content on their own.
EXTRA_STOPS = {
    "thy", "thee", "thou", "thine", "hast", "hath", "doth", "shalt",
    "yea", "nay", "unto", "upon", "therein", "thereof", "thereupon",
    "thus", "said", "came", "come", "went", "like", "also", "even",
    "one", "two", "three", "four", "five", "six", "seven", "ten",
    "him", "his", "her", "hers", "them", "they", "their", "our", "ours",
    "shall", "wilt", "wouldst", "couldst",
}
stop_words = ENGLISH_STOP_WORDS.union(EXTRA_STOPS)

# ── Extract discriminative terms via TF-IDF ────────────────────────────────────
# min_df=5      → term must appear in at least 5 different verses
# max_df=0.15   → term must not appear in more than 15 % of verses
# stop_words    → sklearn built-in English list + biblical extras; filters both
#                 stopword unigrams and any bigram containing a stopword
# ngram_range   → unigrams + bigrams
# max_features  → cap vocabulary before scoring to keep memory reasonable
vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.15,
    max_features=10_000,
    stop_words='english',
    analyzer="word",
    token_pattern=r"[a-zA-Z]{3,}(?:\s[a-zA-Z]{3,})?",
)
tfidf_matrix = vectorizer.fit_transform(verses)   # sparse (n_verses × n_features)

# Score each term by its *maximum* TF-IDF value across all verses.
# .toarray() is required — np.asarray() on a sparse matrix wraps the object
# rather than densifying it, yielding a 1-element object array.
max_tfidf = tfidf_matrix.max(axis=0).toarray().ravel()
vocab_array = np.array(vectorizer.get_feature_names_out())

TOP_N = 3_000
top_idx = np.argsort(max_tfidf)[::-1][:TOP_N]

vocab_df = pd.DataFrame({
    "concept": vocab_array[top_idx],
    "max_tfidf": max_tfidf[top_idx],
})

print(f"\nSelected {len(vocab_df)} terms.  Top 30 by max TF-IDF:")
print(vocab_df.head(30)["concept"].tolist())

# ── Embed all vocabulary terms ─────────────────────────────────────────────────
model = SentenceTransformer("models/greek-nt-sbert_v2")

print("\nEncoding vocabulary...")
embeddings = model.encode(
    vocab_df["concept"].tolist(),
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=128,
)
faiss.normalize_L2(embeddings)

out_dir = "data/greek-nt-sbert_v2"
os.makedirs(out_dir, exist_ok=True)
vocab_df.to_pickle(os.path.join(out_dir, "esv_vocab.pkl"))
np.save(os.path.join(out_dir, "esv_vocab_embeddings.npy"), embeddings)

print(f"\nSaved → {out_dir}/esv_vocab.pkl            ({len(vocab_df)} terms)")
print(f"Saved → {out_dir}/esv_vocab_embeddings.npy  shape={embeddings.shape}")


ESV corpus: 31104 verses

Selected 3000 terms.  Top 30 by max TF-IDF:
['heart will', 'abundance', 'abundantly', 'zadok', 'account', 'zechariah', 'the mighty', 'the men', 'the night', 'the north', 'the nile', 'behold', 'your wrath', 'before him', 'the lowland', 'hezekiah', 'your words', 'ziph', 'afflicted', 'hell', 'frame', 'longer', 'little children', 'man', 'for there', 'makes his', 'and every', 'and for', 'mankind', 'mark']


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2392.99it/s, Materializing param=pooler.dense.weight]                               



Encoding vocabulary...


Batches: 100%|██████████| 24/24 [00:00<00:00, 39.04it/s]


Saved → data/greek-nt-sbert_v2/esv_vocab.pkl            (3000 terms)
Saved → data/greek-nt-sbert_v2/esv_vocab_embeddings.npy  shape=(3000, 768)


In [1]:

# ── BERTopic: Discover Latent NT Concept Themes ───────────────────────────────
# Embeds ESV NT verses with greek-nt-sbert_v2 and runs BERTopic clustering.
# Results are saved for auditing against the hand-curated concept_bank.pkl.

import os
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

NT_BOOKS = [
    "Matthew", "Mark", "Luke", "John", "Acts",
    "Romans", "1 Corinthians", "2 Corinthians", "Galatians", "Ephesians",
    "Philippians", "Colossians", "1 Thessalonians", "2 Thessalonians",
    "1 Timothy", "2 Timothy", "Titus", "Philemon",
    "Hebrews", "James", "1 Peter", "2 Peter",
    "1 John", "2 John", "3 John", "Jude", "Revelation",
]

esv = pd.read_csv("data/esv.csv")
esv_nt = esv[esv["book"].isin(NT_BOOKS)].reset_index(drop=True)
docs = esv_nt["verse_text"].tolist()
print(f"NT verses: {len(docs)} from {esv_nt['book'].nunique()} books")

# Embed NT verses (cached after first run)
emb_cache = "data/greek-nt-sbert_v2/esv_nt_embeddings.npy"
if os.path.exists(emb_cache):
    print("Loading cached NT verse embeddings...")
    embeddings = np.load(emb_cache)
else:
    print("Encoding NT verses with greek-nt-sbert_v2 (this may take a minute)...")
    model = SentenceTransformer("models/greek-nt-sbert_v2")
    embeddings = model.encode(docs, show_progress_bar=True, convert_to_numpy=True, batch_size=64)
    faiss.normalize_L2(embeddings)
    np.save(emb_cache, embeddings)
    print(f"Saved -> {emb_cache}  shape={embeddings.shape}")

# ── BERTopic configuration ─────────────────────────────────────────────────────
# Use a small min_cluster_size so HDBSCAN forms fine-grained clusters
# before BERTopic merges them down to ~50 theological topics.
umap_model = UMAP(
    n_components=10,
    n_neighbors=10,
    min_dist=0.0,
    metric="cosine",
    random_state=42,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=5,
    min_samples=2,
    metric="euclidean",
    cluster_selection_method="eom",
    prediction_data=True,
)
# CountVectorizer strips stop words so c-TF-IDF keywords are meaningful
vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
)
topic_model = BERTopic(
    embedding_model=None,           # pass pre-computed embeddings directly
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer,
    nr_topics=50,                   # merge down to ~50 theological themes
    calculate_probabilities=False,
    verbose=True,
)

print("\nFitting BERTopic on NT verse embeddings...")
topics, _ = topic_model.fit_transform(docs, embeddings)

# ── Save results ───────────────────────────────────────────────────────────────
out_dir = "data/greek-nt-sbert_v2"
topic_info = topic_model.get_topic_info()

print(f"\nTopics found: {len(topic_info) - 1} (excluding outlier cluster -1)")
print(f"Outlier verses: {(np.array(topics) == -1).sum()}")

# Flatten: top-8 keywords + verse count per topic
rows = []
for _, row in topic_info[topic_info["Topic"] != -1].iterrows():
    tid = row["Topic"]
    words = topic_model.get_topic(tid)
    rows.append({
        "topic_id":  tid,
        "count":     int(row["Count"]),
        "top_words": ", ".join(w for w, _ in words[:8]),
    })

terms_df = pd.DataFrame(rows).sort_values("count", ascending=False).reset_index(drop=True)
topic_info.to_pickle(os.path.join(out_dir, "bertopic_topic_info.pkl"))
terms_df.to_pickle(os.path.join(out_dir, "bertopic_terms.pkl"))

print(f"Saved -> {out_dir}/bertopic_topic_info.pkl")
print(f"Saved -> {out_dir}/bertopic_terms.pkl  ({len(terms_df)} topics)")
print("\nAll topics by verse count:")
print(terms_df.to_string(index=False))


/home/jon/Documents/codespace/projects/greek-semantic-ui/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-10 13:55:39,113 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


NT verses: 6229 from 15 books
Loading cached NT verse embeddings...

Fitting BERTopic on NT verse embeddings...


2026-03-10 13:55:52,664 - BERTopic - Dimensionality - Completed ✓
2026-03-10 13:55:52,665 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-10 13:55:52,822 - BERTopic - Cluster - Completed ✓
2026-03-10 13:55:52,822 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-03-10 13:55:52,956 - BERTopic - Representation - Completed ✓
2026-03-10 13:55:52,957 - BERTopic - Topic reduction - Reducing number of topics
2026-03-10 13:55:52,967 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-10 13:55:53,044 - BERTopic - Representation - Completed ✓
2026-03-10 13:55:53,045 - BERTopic - Topic reduction - Reduced number of topics from 406 to 50



Topics found: 49 (excluding outlier cluster -1)
Outlier verses: 1797
Saved -> data/greek-nt-sbert_v2/bertopic_topic_info.pkl
Saved -> data/greek-nt-sbert_v2/bertopic_terms.pkl  (49 topics)

All topics by verse count:
 topic_id  count                                                                                    top_words
        0   1128                                       father, god, heaven, know, let, brothers, say, kingdom
        1    979                                       went, said, came, jesus, saw, peter, disciples, saying
        2    320                    christ, jesus christ, god, lord jesus, grace, christ jesus, spirit, jesus
        3    266                                     law, light, life, sin, god, righteousness, person, faith
        4    198                  pharisees, said, disciples, sabbath, scribes, answered, said disciples, man
        5    197                                        mary, woman, mother, daughter, wife, girl, came, said
        6   

In [2]:

# ── Audit: BERTopic topics vs. hand-curated concept bank ─────────────────────
# For each discovered topic, find its nearest concept in concept_bank.pkl.
# Topics with low similarity are coverage gaps worth reviewing for additions.

import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

concept_bank = pd.read_pickle("data/greek-nt-sbert_v2/concept_bank.pkl")
concept_embs = np.load("data/greek-nt-sbert_v2/concept_embeddings.npy")
terms_df     = pd.read_pickle("data/greek-nt-sbert_v2/bertopic_terms.pkl")

# Encode the leading keyword of each topic into the same embedding space
model = SentenceTransformer("models/greek-nt-sbert_v2")
lead_words = [row["top_words"].split(",")[0].strip() for _, row in terms_df.iterrows()]
topic_embs = model.encode(lead_words, convert_to_numpy=True, batch_size=64)
faiss.normalize_L2(topic_embs)
np.save("data/greek-nt-sbert_v2/bertopic_topic_embeddings.npy", topic_embs)

# Nearest concept in concept bank for each topic
sims = topic_embs @ concept_embs.T          # (n_topics, n_concepts)
best_idx = sims.argmax(axis=1)
best_sim  = sims.max(axis=1)

terms_df = terms_df.copy()
terms_df["nearest_concept"]  = concept_bank.iloc[best_idx]["concept"].values
terms_df["nearest_category"] = concept_bank.iloc[best_idx]["category"].values
terms_df["concept_sim"]      = best_sim.round(4)

# ── Coverage report ────────────────────────────────────────────────────────────
GAP_THRESHOLD = 0.60
covered = terms_df[terms_df["concept_sim"] >= GAP_THRESHOLD].sort_values("count", ascending=False)
gaps    = terms_df[terms_df["concept_sim"] <  GAP_THRESHOLD].sort_values("count", ascending=False)

print(f"BERTopic topics: {len(terms_df)}")
print(f"  Well-covered (sim ≥ {GAP_THRESHOLD}): {len(covered)}")
print(f"  Potential gaps (sim < {GAP_THRESHOLD}): {len(gaps)}")

print("\n─── GAPS ── topics not well covered by current concept bank ───")
print(gaps[["count", "top_words", "nearest_concept", "concept_sim"]].to_string(index=False))

print("\n─── COVERED ── topics mapped to existing concept bank entries ───")
print(covered[["count", "top_words", "nearest_concept", "nearest_category"]].to_string(index=False))

# ── Suggested additions (top-word of each gap topic as candidate concept) ─────
print("\n─── CANDIDATE ADDITIONS (gap topic lead words) ───")
candidates = gaps[["count","top_words"]].copy()
candidates["candidate_concept"] = gaps["top_words"].str.split(",").str[0].str.strip()
print(candidates[["count","candidate_concept","top_words"]].head(30).to_string(index=False))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2416.99it/s, Materializing param=pooler.dense.weight]                               


BERTopic topics: 49
  Well-covered (sim ≥ 0.6): 49
  Potential gaps (sim < 0.6): 0

─── GAPS ── topics not well covered by current concept bank ───
Empty DataFrame
Columns: [count, top_words, nearest_concept, concept_sim]
Index: []

─── COVERED ── topics mapped to existing concept bank entries ───
 count                                                                                    top_words        nearest_concept      nearest_category
  1128                                       father, god, heaven, know, let, brothers, say, kingdom                   hell           Eschatology
   979                                       went, said, came, jesus, saw, peter, disciples, saying                  anger    Sin and Corruption
   320                    christ, jesus christ, god, lord jesus, grace, christ jesus, spirit, jesus  incarnation of Christ           Christology
   266                                     law, light, life, sin, god, righteousness, person, faith fulfillment of the la

In [ ]:

# ── Parse ESV JSON → CSV ──────────────────────────────────────────────────────
# Source: https://github.com/lguenth/mdbible  (json/ESV.json)
# The JSON stores token lists with optional Strong's numbers:
#   verse → [ [word_text, ?strongs_id], ... ]
# We join word_text tokens with a space before alpha-starting tokens
# to reconstruct clean, readable verse text.

import json
import re
import os
import pandas as pd

ESV_JSON = "data/esv_source/ESV.json"   # adjust if cloned elsewhere
ESV_CSV  = "data/esv.csv"

if not os.path.exists(ESV_JSON):
    raise FileNotFoundError(
        f"ESV JSON not found at {ESV_JSON}.\n"
        "Clone the repo and copy json/ESV.json there:\n"
        "  git clone --depth=1 https://github.com/lguenth/mdbible.git /tmp/mdbible\n"
        "  mkdir -p data/esv_source\n"
        "  cp /tmp/mdbible/json/ESV.json data/esv_source/"
    )

def tokens_to_text(tokens: list) -> str:
    """Join ESV JSON token list into readable verse text."""
    parts = []
    for tok in tokens:
        word = tok[0]
        if parts and re.match(r'[A-Za-z\u2018\u2019\u201c\u201d]', word):
            parts.append(' ')
        parts.append(word)
    return ''.join(parts).strip()

with open(ESV_JSON, encoding="utf-8") as f:
    esv_data = json.load(f)

books = esv_data["books"]   # dict: book_name → list[chapters] → list[verse_tokens]

rows = []
for book_name, chapters in books.items():
    for chap_idx, verses in enumerate(chapters, start=1):
        for verse_idx, tokens in enumerate(verses, start=1):
            rows.append({
                "book":        book_name,
                "chapter":     chap_idx,
                "verse":       verse_idx,
                "verse_text":  tokens_to_text(tokens),
            })

esv_df = pd.DataFrame(rows)
esv_df.to_csv(ESV_CSV, index=False)

print(f"ESV corpus: {len(esv_df)} verses across {esv_df['book'].nunique()} books")
print(f"Saved → {ESV_CSV}")
print()
print(esv_df.sample(5)[["book","chapter","verse","verse_text"]].to_string(index=False))


ESV corpus: 31104 verses across 66 books
Saved → data/esv.csv

              book  chapter  verse                                                                                                                                                                                                                                                              verse_text
               Job       38     31                                                                                                                                                                                                    Can you bind the chains of the Pleiades or loose the cords of Orion?
               Job       28     10                                                                                                                                                                                               He cuts out channels in the rocks, and his eye sees every precious thing.
            Esther        9      8      

: 